# ДЗ 2 — Перцептрон Розенблатта в 30 строк

> Если зачем-то закрыли лекцию: это [Модуль 1](https://itrubnikov.github.io/Train_of_Thought/modules/01-history) курса «От нуля до своих агентов», вставка про Mark I.

Сейчас вы соберёте — в браузере, безо всякого PyTorch — то, что Розенблатт собирал в 1958 году из ламп, моторов и потенциометров. Цель: научить однослойный перцептрон отличать треугольник от квадрата на картинке 8×8 пикселей.

**Что от вас требуется:** дописать **две** короткие функции (помечены `TODO`). Это ~5 строк кода суммарно. Остальное уже написано.

**Время:** ~60—90 минут вместе с пониманием.

**Как сдавать:** в Colab — `Файл → Сохранить копию на Диске`, дописать TODO, запустить все ячейки, **расшарить** ноутбук («Поделиться → у кого есть ссылка → Просмотр») и прислать ссылку в чат курса.

## Шаг 0. Настройка

В Colab `numpy` и `matplotlib` уже стоят — ничего ставить не надо. Если запускаете локально (Jupyter / VS Code), раскомментируйте `pip install` в ячейке ниже.

In [ ]:
# !pip install numpy matplotlib  # раскомментируйте для локального запуска
import random
from typing import List, Tuple
import numpy as np
import matplotlib.pyplot as plt

GRID = 8
N_FEATURES = GRID * GRID  # 64 пикселя

print(f'Готово. Размер картинки: {GRID}×{GRID} = {N_FEATURES} входов.')

## Шаг 1. Датасет — треугольники и квадраты

Готовая функция, **запустите** и посмотрите на пару примеров. Картинки идут плоским массивом из 64 нулей и единиц (по одному на пиксель).

- `1` = тёмная клетка (есть закраска),
- `0` = светлая клетка.

In [ ]:
def _blank() -> np.ndarray:
    return np.zeros((GRID, GRID), dtype=np.int8)

def make_triangle(rng: random.Random) -> np.ndarray:
    img = _blank()
    size = rng.randint(4, 6)
    top_row = rng.randint(0, GRID - size)
    top_col = rng.randint(0, GRID - size)
    for r in range(size):
        half = r
        center = top_col + size // 2
        for c in range(center - half, center + half + 1):
            if 0 <= c < GRID:
                img[top_row + r, c] = 1
    return img.flatten()

def make_square(rng: random.Random) -> np.ndarray:
    img = _blank()
    size = rng.randint(3, 5)
    top_row = rng.randint(0, GRID - size)
    top_col = rng.randint(0, GRID - size)
    img[top_row:top_row+size, top_col:top_col+size] = 1
    return img.flatten()

def make_dataset(n_per_class: int, seed: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    rng = random.Random(seed)
    triangles = [make_triangle(rng) for _ in range(n_per_class)]
    squares = [make_square(rng) for _ in range(n_per_class)]
    X = np.array(triangles + squares, dtype=np.float32)
    y = np.array([1] * n_per_class + [0] * n_per_class, dtype=np.float32)
    perm = rng.sample(range(len(X)), len(X))
    return X[perm], y[perm]

# Покажем по одному примеру каждого класса
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
rng = random.Random(0)
axes[0].imshow(make_triangle(rng).reshape(GRID, GRID), cmap='Purples')
axes[0].set_title('треугольник (y=1)'); axes[0].axis('off')
axes[1].imshow(make_square(rng).reshape(GRID, GRID), cmap='Purples')
axes[1].set_title('квадрат (y=0)'); axes[1].axis('off')
plt.show()

## Шаг 2. Перцептрон — ваш код

Класс `Perceptron` — это всё, что у нас «учится». В нём один массив весов длиной 64 (по одному весу на пиксель) и один шаг обучения.

**Ниже две `TODO`. Каждая — буквально одна строка.** Подсказки в комментариях.

In [ ]:
class Perceptron:
    def __init__(self, n_features: int = N_FEATURES, lr: float = 0.1):
        self.weights: np.ndarray = np.zeros(n_features, dtype=np.float32)
        self.lr: float = lr  # шаг обучения (как сильно крутить «потенциометры» за раз)

    # ----- TODO 1 -----------------------------------------------------------
    # predict(x): вернуть 1, если взвешенная сумма > 0, иначе 0.
    #
    # Это ровно то, что делал Mark I:
    #   sum_input = w · x   (np.dot(self.weights, x))
    #   output    = 1 если sum_input > 0, иначе 0
    # ------------------------------------------------------------------------
    def predict(self, x: np.ndarray) -> int:
        # PASTE: одну строку.
        raise NotImplementedError('Реализуй predict — см. TODO 1.')

    # ----- TODO 2 -----------------------------------------------------------
    # train_step(x, y_true): один шаг правила перцептрона (электромоторы Mark I).
    #
    #   y_pred = self.predict(x)
    #   error  = y_true - y_pred       # 0, +1 или -1
    #   self.weights += self.lr * error * x
    # ------------------------------------------------------------------------
    def train_step(self, x: np.ndarray, y_true: float) -> None:
        # PASTE: реализуй три строки выше.
        raise NotImplementedError('Реализуй train_step — см. TODO 2.')

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> float:
        preds = np.array([self.predict(x) for x in X])
        return float((preds == y).mean())

## Шаг 3. Тренировка и замер

Готовый код. Просто запустите — он использует ваш `Perceptron`.

In [ ]:
def train_loop(p: Perceptron, X_tr, y_tr, X_te, y_te, epochs: int = 200) -> List[float]:
    history = []
    for _ in range(epochs):
        for x, y in zip(X_tr, y_tr):
            p.train_step(x, y)
        history.append(p.evaluate(X_te, y_te))
    return history

def plot_history(history, title: str):
    plt.figure(figsize=(7, 4))
    plt.plot(history, color='#7f5cca', linewidth=2)
    plt.axhline(0.9, color='#10b981', linestyle='--', alpha=0.6, label='цель: 90%')
    plt.xlabel('эпоха'); plt.ylabel('точность на тесте')
    plt.title(title); plt.ylim(0.4, 1.02); plt.grid(alpha=0.3); plt.legend()
    plt.tight_layout(); plt.show()

## Шаг 4. Эксперимент 1 — треугольник vs квадрат

Цель: точность **≥ 90%**. Если у вас > 90% и кривая на графике растёт — TODO реализованы правильно.

In [ ]:
X_tr, y_tr = make_dataset(n_per_class=80, seed=42)
X_te, y_te = make_dataset(n_per_class=40, seed=7)

p = Perceptron()
history = train_loop(p, X_tr, y_tr, X_te, y_te, epochs=200)

print(f'финальная точность на тесте: {history[-1]:.1%}')
plot_history(history, 'Перцептрон Розенблатта: треугольник vs квадрат')

## Бонус: тот самый XOR-потолок 1969 года

В лекции упоминается, что Минский и Пейперт в книге _Perceptrons_ (1969) формально показали: **однослойный** перцептрон не может выучить XOR-подобные задачи. Из-за этого нейросети «умерли» на 17 лет до изобретения backpropagation в 1986-м.

Ниже датасет, который намеренно построен XOR-подобно: классы **не разделимы** одной линейной границей. Тот же ваш `Perceptron`, тот же цикл обучения — но точность застрянет на ~50% (т. е. наугад). Вот он, тот самый потолок 1969 года, в исполнении вашего собственного кода.

In [ ]:
def make_xor_like_dataset(n_per_class: int, seed: int = 0):
    rng = random.Random(seed)
    X, y = [], []
    def shifted(shape: str, dr: int, dc: int):
        img = _blank(); size = 3
        for r in range(size):
            for c in range(size):
                if shape == 'triangle' and c <= r:
                    img[dr+r, dc+c] = 1
                elif shape == 'square':
                    img[dr+r, dc+c] = 1
        return img.flatten()
    for _ in range(n_per_class):
        X.append(shifted('triangle', 0, 0) if rng.random() < 0.5 else shifted('square', 5, 5))
        y.append(1)
        X.append(shifted('triangle', 5, 5) if rng.random() < 0.5 else shifted('square', 0, 0))
        y.append(0)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_tr2, y_tr2 = make_xor_like_dataset(n_per_class=80, seed=42)
X_te2, y_te2 = make_xor_like_dataset(n_per_class=40, seed=7)

p2 = Perceptron()
history2 = train_loop(p2, X_tr2, y_tr2, X_te2, y_te2, epochs=200)

print(f'финальная точность: {history2[-1]:.1%}  ← около 50% и есть потолок 1969-го')
plot_history(history2, 'XOR-подобная задача: тот самый потолок 1969-го')

## Что вы только что сделали

В 30 строках Python воспроизведена та же машина, которую Розенблатт собирал из ламп и моторов в 1958 году. Тот же алгоритм, те же ограничения, тот же XOR-потолок, который зафиксировали Минский и Пейперт в 1969-м.

Чтобы пробить этот потолок, нужен ещё один обучаемый слой и алгоритм обратного распространения ошибки. К этому подойдём в [Модуле 5.5 «Как LLM думает»](https://itrubnikov.github.io/Train_of_Thought/modules/05-5-llm-mental-model) и далее.

**Как сдать:**
1. `Файл → Сохранить копию на Диске`.
2. `Поделиться → у кого есть ссылка → Просмотр`.
3. Прислать ссылку в чат курса.

**Чек перед сдачей:**
- [ ] Эксперимент 1: точность ≥ 90% на тесте.
- [ ] Эксперимент 2: точность ~50% (это **правильный** результат, не баг).
- [ ] Оба графика отрисовались.